In [1]:
import ast
import glob
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

In [2]:
ROOT = Path.cwd()
if not (ROOT / "summarized_results").exists() and (ROOT.parent / "summarized_results").exists():
    ROOT = ROOT.parent

OUT = ROOT / "summarized_results" / "supplement_etables"
OUT.mkdir(parents=True, exist_ok=True)

POWER_INPUT_CANDIDATES = [
    ROOT / "summarized_results" / "power_simulation_outputs" / "power_simulation_raw.csv",
    ROOT / "pooled" / "power_simulation_outputs" / "power_simulation_raw.csv",
    ROOT / "power_simulation_outputs" / "power_simulation_raw.csv",
]

CSV = {
    "HYPERION": [ROOT / "hyperion" / "predictorsDf.csv"],
    "eICU": [ROOT / "eICU" / "eICUPredictorsDiag.csv", ROOT / "eICU" / "eICUPredictors.csv"],
    "PMAP": [ROOT / "pmap" / "PMAP_Predictors2.csv", ROOT / "pmap" / "PMAP_Predictors.csv"],
    "MIMIC-IV": [ROOT / "mimiciv" / "MIMIC_Predictors.csv", ROOT / "mimiciv" / "MIMIC_Predictors1.csv"],
}

DML_NOTEBOOK = {
    "HYPERION": ROOT / "hyperion" / "hyperionAnalysisDML.ipynb",
    "eICU": ROOT / "eICU" / "eICUAnalysisDML.ipynb",
    "PMAP": ROOT / "pmap" / "PMAPAnalysisDML.ipynb",
    "MIMIC-IV": ROOT / "mimiciv" / "MIMICAnalysisDML.ipynb",
}

TREATMENT_CANDIDATES = {
    "HYPERION": ["groupe", "TTM", "hypothermia"],
    "eICU": ["Hypothermia", "hypothermia", "treatment_hypothermia"],
    "PMAP": ["hypothermia", "TTM"],
    "MIMIC-IV": ["hypothermia", "TTM"],
}

OUTCOME_OR_ID_COLUMNS = {
    "patientunitstayid", "subject_id", "hadm_id", "stay_id", "osler_id", "pat_enc_csn_id", "SUBJID",
    "death_at_disch", "DeathAtDischarge", "hospital_mortality", "hospital_expire_flag",
    "LastMGCSPositive", "last_mGCS", "last_mGCS_time", "first_mGCS_time",
    "LastGCSPositive", "LastGCS15", "LastGCS", "LastMGCSTime", "FirstMGCSTime",
    "CPC_SC3", "CPC12", "BARTHEL_SC", "SOFA_SC7", "DS_DC", "DAYS_ALIVE_30",
}

print("Repository root:", ROOT)
print("Output directory:", OUT)

Repository root: /home/mbranda1/ttmhte
Output directory: /home/mbranda1/ttmhte/summarized_results/supplement_etables


In [3]:
def first_existing(paths):
    for path in paths:
        if Path(path).exists():
            return Path(path)
    return None


def find_col(df, candidates):
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None


def parse_first_columns_list(ipynb_path):
    if not Path(ipynb_path).exists():
        return []
    import nbformat
    nb = nbformat.read(ipynb_path, as_version=4)
    for cell in nb.cells:
        if cell.cell_type != "code":
            continue
        src = cell.source
        m = re.search(r"columns\s*=\s*(\[.*?\])", src, flags=re.S)
        if not m:
            continue
        try:
            val = ast.literal_eval(m.group(1))
        except Exception:
            continue
        if isinstance(val, list) and val:
            return [str(x) for x in val]
    return []


def normalize_treatment(series, dataset):
    s = series.copy()
    if dataset == "HYPERION":
        if s.dtype == object:
            txt = s.astype(str).str.strip().str.lower()
            return txt.isin(["1", "ttm", "hypothermia", "33", "33c", "yes", "true"]).astype(int)
        return pd.to_numeric(s, errors="coerce").fillna(0).astype(int)
    if s.dtype == object:
        txt = s.astype(str).str.strip().str.lower()
        return txt.isin(["1", "yes", "true", "t", "ttm", "hypothermia"]).astype(int)
    return (pd.to_numeric(s, errors="coerce").fillna(0) != 0).astype(int)


def format_missing(n_missing, denom):
    pct = 100 * n_missing / denom if denom else np.nan
    return f"{int(n_missing)}/{int(denom)} ({pct:.1f}%)" if denom else "0/0 (NA)"


def missingness_test(miss, treatment):
    tab = pd.crosstab(treatment, miss).reindex(index=[0, 1], columns=[False, True], fill_value=0)
    arr = tab.to_numpy()
    if arr.sum() == 0 or arr.shape != (2, 2):
        return np.nan, "not tested"
    try:
        _, p_chi, _, expected = chi2_contingency(arr, correction=False)
        if (expected < 5).any():
            _, p = fisher_exact(arr)
            return float(p), "Fisher exact"
        return float(p_chi), "chi-square"
    except Exception:
        return np.nan, "not tested"

## eTables 7-14: Power Simulation Tables

In [4]:
power_input = first_existing(POWER_INPUT_CANDIDATES)
power_tables = {}
if power_input is None:
    print("No power_simulation_raw.csv found. Run summarized_results/powerSimulationAnalysis.ipynb first, then rerun this notebook.")
    power_raw = pd.DataFrame()
else:
    print("Loaded power input:", power_input)
    power_raw = pd.read_csv(power_input)
    display(power_raw.head())

def make_power_public_table(sub):
    out = sub.copy()
    rename = {
        "dataset": "Dataset",
        "outcome": "Outcome",
        "n": "N",
        "treat_prev": "TTM prevalence",
        "subgroup_prev": "Subgroup prevalence",
        "baseline_risk": "Baseline outcome risk",
        "abs_treatment_effect": "Absolute subgroup treatment effect",
        "n_sims": "Simulations",
        "n_successful_fits": "Successful fits",
        "n_failed_fits": "Failed fits",
        "power": "Power",
    }
    keep = [c for c in rename if c in out.columns]
    out = out[keep].rename(columns=rename)
    for col in ["TTM prevalence", "Subgroup prevalence", "Baseline outcome risk", "Absolute subgroup treatment effect", "Power"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce").round(3)
    for col in ["N", "Simulations", "Successful fits", "Failed fits"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype("Int64")
    if "Failed fits" in out.columns and out["Failed fits"].fillna(0).eq(0).all():
        out = out.drop(columns=["Failed fits"])
    return out

if not power_raw.empty:
    order = [
        ("HYPERION", "mortality"),
        ("HYPERION", "neuro"),
        ("eICU", "mortality"),
        ("eICU", "neuro"),
        ("PMAP", "mortality"),
        ("PMAP", "neuro"),
        ("MIMIC-IV", "mortality"),
        ("MIMIC-IV", "neuro"),
    ]
    for i, (dataset, outcome) in enumerate(order, start=7):
        sub = power_raw[
            power_raw["dataset"].astype(str).eq(dataset)
            & power_raw["outcome"].astype(str).eq(outcome)
        ].copy()
        table = make_power_public_table(sub)
        name = f"eTable{i:02d}_power_{dataset.lower().replace('-', '').replace(' ', '_')}_{outcome}"
        power_tables[name] = table
        table.to_csv(OUT / f"{name}.csv", index=False)
        print(name, table.shape)
        display(table)

Loaded power input: /home/mbranda1/ttmhte/summarized_results/power_simulation_outputs/power_simulation_raw.csv


,dataset,outcome,n,treat_prev,subgroup_prev,baseline_risk,abs_treatment_effect,n_sims,n_successful_fits,n_failed_fits,power
0,HYPERION,mortality,581,0.488812,0.2,0.83165,0.10,2000,1987,13,0.358832
1,HYPERION,mortality,581,0.488812,0.2,0.83165,0.15,2000,1873,127,0.802456
2,HYPERION,mortality,581,0.488812,0.2,0.83165,0.20,2000,1756,244,0.992597
3,HYPERION,mortality,581,0.488812,0.3,0.83165,0.10,2000,2000,0,0.444500
4,HYPERION,mortality,581,0.488812,0.3,0.83165,0.15,2000,1918,82,0.920229


eTable07_power_hyperion_mortality (9, 11)


,Dataset,Outcome,N,TTM prevalence,Subgroup prevalence,Baseline outcome risk,Absolute subgroup treatment effect,Simulations,Successful fits,Failed fits,Power
0,HYPERION,mortality,581,0.489,0.2,0.832,0.10,2000,1987,13,0.359
1,HYPERION,mortality,581,0.489,0.2,0.832,0.15,2000,1873,127,0.802
2,HYPERION,mortality,581,0.489,0.2,0.832,0.20,2000,1756,244,0.993
3,HYPERION,mortality,581,0.489,0.3,0.832,0.10,2000,2000,0,0.444
4,HYPERION,mortality,581,0.489,0.3,0.832,0.15,2000,1918,82,0.920
5,HYPERION,mortality,581,0.489,0.3,0.832,0.20,2000,1815,185,0.999
6,HYPERION,mortality,581,0.489,0.5,0.832,0.10,2000,2000,0,0.538
7,HYPERION,mortality,581,0.489,0.5,0.832,0.15,2000,1978,22,0.977
8,HYPERION,mortality,581,0.489,0.5,0.832,0.20,2000,1629,371,1.000


eTable08_power_hyperion_neuro (9, 11)


...

,Dataset,Outcome,N,TTM prevalence,Subgroup prevalence,Baseline outcome risk,Absolute subgroup treatment effect,Simulations,Successful fits,Power
27,PMAP,neuro,1289,0.48,0.2,0.319,0.10,2000,2000,0.331
28,PMAP,neuro,1289,0.48,0.2,0.319,0.15,2000,2000,0.585
29,PMAP,neuro,1289,0.48,0.2,0.319,0.20,2000,2000,0.816
30,PMAP,neuro,1289,0.48,0.3,0.319,0.10,2000,2000,0.402
31,PMAP,neuro,1289,0.48,0.3,0.319,0.15,2000,2000,0.708
32,PMAP,neuro,1289,0.48,0.3,0.319,0.20,2000,2000,0.906
33,PMAP,neuro,1289,0.48,0.5,0.319,0.10,2000,2000,0.452
34,PMAP,neuro,1289,0.48,0.5,0.319,0.15,2000,2000,0.780
35,PMAP,neuro,1289,0.48,0.5,0.319,0.20,2000,2000,0.946


eTable13_power_mimiciv_mortality (9, 10)


,Dataset,Outcome,N,TTM prevalence,Subgroup prevalence,Baseline outcome risk,Absolute subgroup treatment effect,Simulations,Successful fits,Power
54,MIMIC-IV,mortality,611,0.458,0.2,0.671,0.10,2000,2000,0.200
55,MIMIC-IV,mortality,611,0.458,0.2,0.671,0.15,2000,2000,0.415
56,MIMIC-IV,mortality,611,0.458,0.2,0.671,0.20,2000,2000,0.679
57,MIMIC-IV,mortality,611,0.458,0.3,0.671,0.10,2000,2000,0.258
58,MIMIC-IV,mortality,611,0.458,0.3,0.671,0.15,2000,2000,0.537
59,MIMIC-IV,mortality,611,0.458,0.3,0.671,0.20,2000,2000,0.805
60,MIMIC-IV,mortality,611,0.458,0.5,0.671,0.10,2000,2000,0.290
61,MIMIC-IV,mortality,611,0.458,0.5,0.671,0.15,2000,2000,0.596
62,MIMIC-IV,mortality,611,0.458,0.5,0.671,0.20,2000,2000,0.880


eTable14_power_mimiciv_neuro (9, 10)


,Dataset,Outcome,N,TTM prevalence,Subgroup prevalence,Baseline outcome risk,Absolute subgroup treatment effect,Simulations,Successful fits,Power
63,MIMIC-IV,neuro,560,0.5,0.2,0.336,0.10,2000,2000,0.158
64,MIMIC-IV,neuro,560,0.5,0.2,0.336,0.15,2000,2000,0.306
65,MIMIC-IV,neuro,560,0.5,0.2,0.336,0.20,2000,2000,0.464
66,MIMIC-IV,neuro,560,0.5,0.3,0.336,0.10,2000,2000,0.188
67,MIMIC-IV,neuro,560,0.5,0.3,0.336,0.15,2000,2000,0.378
68,MIMIC-IV,neuro,560,0.5,0.3,0.336,0.20,2000,2000,0.594
69,MIMIC-IV,neuro,560,0.5,0.5,0.336,0.10,2000,2000,0.225
70,MIMIC-IV,neuro,560,0.5,0.5,0.336,0.15,2000,2000,0.418
71,MIMIC-IV,neuro,560,0.5,0.5,0.336,0.20,2000,2000,0.644


## eTables 16-19: Missingness by TTM Group

In [5]:
missingness_public_tables = {}
missingness_audit_tables = {}

missing_order = [
    ("HYPERION", 16),
    ("eICU", 17),
    ("PMAP", 18),
    ("MIMIC-IV", 19),
]

for dataset, etable_num in missing_order:
    csv_path = first_existing(CSV[dataset])
    if csv_path is None:
        print(f"{dataset}: no predictor CSV found; skipping eTable {etable_num}.")
        continue
    df = pd.read_csv(csv_path)
    treat_col = find_col(df, TREATMENT_CANDIDATES[dataset])
    if treat_col is None:
        print(f"{dataset}: no treatment column found; skipping eTable {etable_num}.")
        continue
    treatment = normalize_treatment(df[treat_col], dataset)

    dml_cols = parse_first_columns_list(DML_NOTEBOOK[dataset])
    if dataset == "HYPERION" or not dml_cols:
        feature_cols = [
            c for c in df.columns
            if c not in OUTCOME_OR_ID_COLUMNS
            and c != treat_col
            and not c.lower().startswith("unnamed")
        ]
    else:
        feature_cols = [
            c for c in dml_cols
            if c in df.columns
            and c not in OUTCOME_OR_ID_COLUMNS
            and c != treat_col
            and not c.lower().startswith("unnamed")
        ]

    rows = []
    for col in feature_cols:
        miss = df[col].isna()
        no_ttm = treatment.eq(0)
        ttm = treatment.eq(1)
        p, test = missingness_test(miss, treatment)
        significant = bool(pd.notna(p) and p < 0.05)
        rows.append({
            "Variable": col,
            "No TTM missing": format_missing(miss[no_ttm].sum(), no_ttm.sum()),
            "TTM missing": format_missing(miss[ttm].sum(), ttm.sum()),
            "Difference": "*" if significant else "",
            "p_value": p,
            "test": test,
            "n_no_ttm": int(no_ttm.sum()),
            "n_ttm": int(ttm.sum()),
        })

    audit = pd.DataFrame(rows)
    public = audit[["Variable", "No TTM missing", "TTM missing", "Difference"]].copy()
    name = f"eTable{etable_num:02d}_missingness_{dataset.lower().replace('-', '').replace(' ', '_')}"
    public.to_csv(OUT / f"{name}.csv", index=False)
    audit.to_csv(OUT / f"{name}_audit_with_p_values.csv", index=False)
    missingness_public_tables[name] = public
    missingness_audit_tables[f"{name}_audit"] = audit
    print(name, public.shape, "source:", csv_path)
    display(public.head(40))

legend = pd.DataFrame({
    "Legend": [
        "Asterisk denotes a significant difference in the proportion of missing values between TTM and no-TTM groups (chi-square test, or Fisher's exact test when any expected cell count < 5; alpha = 0.05).",
        "The audit files include p_value and test columns for verification only; the public eTables omit those columns.",
    ]
})
legend.to_csv(OUT / "eTables16_19_missingness_legend.csv", index=False)
display(legend)

eTable16_missingness_hyperion (161, 4) source: /home/mbranda1/ttmhte/hyperion/predictorsDf.csv


,Variable,No TTM missing,TTM missing,Difference
0,J0_SEX,0/297 (0.0%),0/284 (0.0%),
1,J0_TAILLE,22/297 (7.4%),30/284 (10.6%),
2,J0_POIDS,2/297 (0.7%),1/284 (0.4%),
3,J0_BMI,24/297 (8.1%),30/284 (10.6%),
4,J0_AGE,2/297 (0.7%),2/284 (0.7%),
5,J0_PAS,1/297 (0.3%),0/284 (0.0%),
6,J0_PAD,1/297 (0.3%),1/284 (0.4%),
7,J0_PAM,2/297 (0.7%),0/284 (0.0%),
8,J0_FC,1/297 (0.3%),0/284 (0.0%),
9,J0_SPO2,6/297 (2.0%),2/284 (0.7%),


/local/mbranda1/3981445/ipykernel_4118070/1622850396.py:16: DtypeWarning: Columns (2188,2190) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


eTable17_missingness_eicu (38, 4) source: /home/mbranda1/ttmhte/eICU/eICUPredictorsDiag.csv


,Variable,No TTM missing,TTM missing,Difference
0,gender,0/2420 (0.0%),0/752 (0.0%),
1,age,0/2420 (0.0%),0/752 (0.0%),
2,bmi,145/2420 (6.0%),7/752 (0.9%),*
3,nurse_first_Non-Invasive BP Systolic,155/2420 (6.4%),23/752 (3.1%),*
4,nurse_first_Non-Invasive BP Diastolic,155/2420 (6.4%),23/752 (3.1%),*
5,nurse_first_Non-Invasive BP Mean,294/2420 (12.1%),51/752 (6.8%),*
6,nurse_first_Heart Rate,105/2420 (4.3%),0/752 (0.0%),*
7,nurse_first_O2 Saturation,251/2420 (10.4%),49/752 (6.5%),*
8,nurse_first_Temperature (C),180/2420 (7.4%),3/752 (0.4%),*
9,lab_first_Respiratory Rate,2081/2420 (86.0%),628/752 (83.5%),


eTable18_missingness_pmap (36, 4) source: /home/mbranda1/ttmhte/pmap/PMAP_Predictors2.csv


,Variable,No TTM missing,TTM missing,Difference
0,gender,0/1044 (0.0%),0/697 (0.0%),
1,age,0/1044 (0.0%),0/697 (0.0%),
2,first_mGCS,0/1044 (0.0%),0/697 (0.0%),
3,flo_first_r_cpn_glasgow_coma_scale_score,186/1044 (17.8%),89/697 (12.8%),*
4,flo_first_bp_systolic,27/1044 (2.6%),0/697 (0.0%),*
5,flo_first_bp_diastolic,27/1044 (2.6%),0/697 (0.0%),*
6,flo_first_r_map,100/1044 (9.6%),49/697 (7.0%),
7,flo_first_temperature,83/1044 (8.0%),3/697 (0.4%),*
8,flo_first_r_ed_pre-arrival_pulse_(heart_rate),912/1044 (87.4%),609/697 (87.4%),
9,flo_first_r_fio2,432/1044 (41.4%),171/697 (24.5%),*


eTable19_missingness_mimiciv (325, 4) source: /home/mbranda1/ttmhte/mimiciv/MIMIC_Predictors.csv


,Variable,No TTM missing,TTM missing,Difference
0,chart_seizure_duration_Status,152/463 (32.8%),86/280 (30.7%),
1,chart_cervical_collar_status_On,152/463 (32.8%),86/280 (30.7%),
2,med_sodium_bicarbonate,152/463 (32.8%),86/280 (30.7%),
3,input_sodium_bicarbonate_8.4%,152/463 (32.8%),86/280 (30.7%),
4,med_calcium_acetate,152/463 (32.8%),86/280 (30.7%),
5,input_calcium_gluconate,152/463 (32.8%),86/280 (30.7%),
6,med_miconazole_2%_cream,152/463 (32.8%),86/280 (30.7%),
7,chart_first_creatinine_(serum),84/463 (18.1%),9/280 (3.2%),*
8,chart_first_d-dimer,458/463 (98.9%),279/280 (99.6%),
9,output_first_d-dimer,458/463 (98.9%),279/280 (99.6%),


,Legend
0,Asterisk denotes a significant difference in t...
1,The audit files include p_value and test colum...


## Combined Workbook

In [6]:
workbook = OUT / "supplement_etables_public.xlsx"
audit_workbook = OUT / "supplement_etables_audit.xlsx"

def safe_sheet(name):
    return re.sub(r"[^A-Za-z0-9_]", "_", name)[:31]

with pd.ExcelWriter(workbook) as writer:
    wrote = False
    for name, table in {**power_tables, **missingness_public_tables}.items():
        table.to_excel(writer, sheet_name=safe_sheet(name), index=False)
        wrote = True
    if not wrote:
        pd.DataFrame({"message": ["No tables were generated. Check input CSV availability."]}).to_excel(writer, sheet_name="README", index=False)

with pd.ExcelWriter(audit_workbook) as writer:
    wrote = False
    for name, table in missingness_audit_tables.items():
        table.to_excel(writer, sheet_name=safe_sheet(name), index=False)
        wrote = True
    if not wrote:
        pd.DataFrame({"message": ["No audit tables were generated. Check input CSV availability."]}).to_excel(writer, sheet_name="README", index=False)

print("Wrote", workbook)
print("Wrote", audit_workbook)
print("Output files:")
for path in sorted(OUT.glob("*")):
    print(" -", path.name)

Wrote /home/mbranda1/ttmhte/summarized_results/supplement_etables/supplement_etables_public.xlsx
Wrote /home/mbranda1/ttmhte/summarized_results/supplement_etables/supplement_etables_audit.xlsx
Output files:
 - eTable07_power_hyperion_mortality.csv
 - eTable08_power_hyperion_neuro.csv
 - eTable09_power_eicu_mortality.csv
 - eTable10_power_eicu_neuro.csv
 - eTable11_power_pmap_mortality.csv
 - eTable12_power_pmap_neuro.csv
 - eTable13_power_mimiciv_mortality.csv
 - eTable14_power_mimiciv_neuro.csv
 - eTable16_missingness_hyperion.csv
 - eTable16_missingness_hyperion_audit_with_p_values.csv
 - eTable17_missingness_eicu.csv
 - eTable17_missingness_eicu_audit_with_p_values.csv
 - eTable18_missingness_pmap.csv
 - eTable18_missingness_pmap_audit_with_p_values.csv
 - eTable19_missingness_mimiciv.csv
 - eTable19_missingness_mimiciv_audit_with_p_values.csv
 - eTables16_19_missingness_legend.csv
 - supplement_etables_audit.xlsx
 - supplement_etables_public.xlsx
